# STQD6324 Assignment 2: MovieLens Data Management Pipeline

This notebook is my documented version of the MovieLens 100k data management pipeline.

The main technologies used in this assignment are:

- Apache Spark
- HDFS
- Cassandra
- HBase optional extension

The MovieLens files used are:

- `u.data`
- `u.user`
- `u.item`

I also provide the terminal script version:

```text
assignment2_movielens.py
```

The Python script is the main executable version that I ran in the Hortonworks Sandbox terminal using `spark-submit`.

This notebook follows the same pipeline logic, but it is written in a clearer step-by-step form with explanations, code cells, screenshot references, result interpretation, and reproducibility notes.

The overall workflow is:

```text
HDFS raw files
→ Spark RDDs
→ Spark DataFrames
→ data cleaning
→ analytical tasks
→ Cassandra storage
→ Cassandra read-back validation
→ HBase optional extension
```

I kept the code style close to the classroom Spark and Cassandra examples. For example, the pipeline first reads raw text files as RDDs, parses them into `Row` objects, converts them into Spark DataFrames, and then writes the processed results into Cassandra.

## 0. How to Use This Notebook

This notebook is designed to be used in the **Hortonworks Sandbox VM** or an equivalent environment with Spark, HDFS, Cassandra, and the Spark-Cassandra connector configured.

It is **not intended to be run directly in Google Colab**. Colab does not contain the HDFS path, Cassandra service, or HBase environment used in this assignment.

For exact reproduction in my VM, I used the Python script:

```bash
spark-submit --master local[*] \
--conf spark.eventLog.enabled=false \
--conf spark.ui.showConsoleProgress=false \
--packages com.datastax.spark:spark-cassandra-connector_2.11:2.3.0 \
assignment2_movielens.py
```

This notebook is included because the assignment asks for a well-documented Jupyter Notebook. It explains the same pipeline step by step.

In short:

```text
assignment2_movielens.py       main executable script in the VM terminal
assignment2_notebook.ipynb     documented notebook version for review and explanation
```

If the notebook is opened on GitHub, it can be used to review the workflow, code, explanations, and screenshot references. The actual successful terminal run is supported by:

```text
logs/assignment2_run_output.txt
screenshots/
```

## 1. Environment

This assignment was developed and tested in the Hortonworks Sandbox virtual machine.

```text
User account: maria_dev
Python version: 2.7.5
Apache Spark version: 2.3.0.2.6.5.0-292
Hadoop version: 2.7.3.2.6.5.0-292
HBase version: 1.1.2.2.6.5.0-292
CQLSH version: 5.0.1
Dataset location in HDFS: /user/maria_dev/ml-100k/
```

Before running the Spark-Cassandra part, Cassandra must be started and tested with:

```bash
cqlsh 127.0.0.1 9042
```

For the HBase optional extension, HBase should also be running in Ambari. In my case, HBase worked normally after HMaster and HRegionServer were both live.

The environment information is also recorded in:

```text
logs/01_versions.txt
screenshots/02_versions.png
```

In [ ]:
#1.import libraries
from __future__ import print_function

import os
import sys

# For notebook mode, load the Spark-Cassandra connector before Spark starts.
# In terminal mode, I used spark-submit --packages instead.
os.environ["PYSPARK_SUBMIT_ARGS"] = "--packages com.datastax.spark:spark-cassandra-connector_2.11:2.3.0 pyspark-shell"

from pyspark.sql import SparkSession, Row
from pyspark.sql import functions as fn
from pyspark.sql.window import Window

try:
    unicode
except NameError:
    unicode = str

print("Python version:", sys.version)

## 2. Dataset and HDFS Preparation

The MovieLens 100k dataset was uploaded into HDFS before running Spark.

The HDFS path used in this assignment is:

```text
/user/maria_dev/ml-100k/
```

The three required files are:

```text
u.data
u.user
u.item
```

I checked the files using commands such as:

```bash
hdfs dfs -ls /user/maria_dev/ml-100k/
hdfs dfs -cat /user/maria_dev/ml-100k/u.data | head
hdfs dfs -cat /user/maria_dev/ml-100k/u.user | head
hdfs dfs -cat /user/maria_dev/ml-100k/u.item | head
```

The HDFS file check is important because this assignment requires loading and parsing the MovieLens files from HDFS.

The screenshot below records the HDFS file check, file size, row count, and sample data:

```text
screenshots/01_hdfs_files.png
```

![HDFS files](screenshots/01_hdfs_files.png)

In [ ]:
#2.basic path
hdfs_path = "hdfs:///user/maria_dev/ml-100k"
ks = "movielens"

rate_path = hdfs_path + "/u.data"
mv_path = hdfs_path + "/u.item"
usr_path = hdfs_path + "/u.user"

genre_lst = [
    "unknown", "Action", "Adventure", "Animation", "Children",
    "Comedy", "Crime", "Documentary", "Drama", "Fantasy",
    "Film-Noir", "Horror", "Musical", "Mystery", "Romance",
    "Sci-Fi", "Thriller", "War", "Western"
]

print("hdfs path:", hdfs_path)
print("cassandra keyspace:", ks)

## 3. Cassandra Schema Preparation

The Cassandra schema is stored in:

```text
cql/schema.cql
```

It creates the keyspace:

```text
movielens
```

and five result tables:

```text
average_movie_ratings
top10_movies
user_favourite_genres
users_under_20
scientists_30_40
```

In the VM terminal, I used this command to create the schema:

```bash
cqlsh 127.0.0.1 9042 -f cql/schema.cql
```

I kept the schema in a separate `.cql` file because it makes the Cassandra part easier to check and reproduce.

Evidence screenshots:

```text
screenshots/03_cassandra_connection.png
screenshots/04_cassandra_tables.png
```

One thing I needed to pay attention to was that Cassandra must be running before Spark writes data into it. Otherwise, Spark may fail when it tries to connect to `127.0.0.1:9042`.

In [ ]:
# Optional command if this notebook is run inside the VM with cqlsh available.
# In my actual run, I executed this in PuTTY / terminal.

# !cqlsh 127.0.0.1 9042 -f cql/schema.cql

## 4. Parse Functions

This part follows the classroom Spark style.

The basic idea is:

1. Read raw text from HDFS.
2. Use a function to split each line.
3. Return a `Row` object.
4. Convert the RDD of `Row` objects into a Spark DataFrame.

The `u.data` file uses tab separation.

The `u.user` and `u.item` files use pipe `|` separation.

The `u.item` file is slightly special because it contains 19 genre flag columns. For Task 3, I need a movie-genre table, so I convert each movie into one or more genre rows.

In [ ]:
#3.fix text
def fix_txt(x):
    if x is None:
        return ""

    try:
        if isinstance(x, unicode):
            return x
        return x.decode("ISO-8859-1", "ignore")
    except:
        return str(x)


#4.get rating row from u.data
def get_rate(line):
    try:
        p = line.split("\t")

        return Row(
            user_id=int(p[0]),
            movie_id=int(p[1]),
            rating=float(p[2]),
            ts=int(p[3])
        )
    except:
        return None


#5.get movie row from u.item
def get_mv(line):
    try:
        p = line.split("|")

        return Row(
            movie_id=int(p[0]),
            title=fix_txt(p[1])
        )
    except:
        return None


#6.get user row from u.user
def get_usr(line):
    try:
        p = line.split("|")

        return Row(
            user_id=int(p[0]),
            age=int(p[1]),
            gender=fix_txt(p[2]),
            occupation=fix_txt(p[3]),
            zip=fix_txt(p[4])
        )
    except:
        return None


#7.get movie genre row
def get_mg(line):
    out = []

    try:
        p = line.split("|")
        mid = int(p[0])
        ttl = fix_txt(p[1])

        for i in range(len(genre_lst)):
            idx = 5 + i

            if p[idx].strip() == "1":
                out.append(Row(movie_id=mid, title=ttl, genre=genre_lst[i]))

        if len(out) == 0:
            out.append(Row(movie_id=mid, title=ttl, genre="unknown"))

    except:
        pass

    return out

## 5. Start Spark Session

This part creates the Spark session and connects Spark with Cassandra.

Cassandra is accessed through:

```text
127.0.0.1:9042
```

because Cassandra was running inside the same sandbox VM.

In my final run, I used local Spark mode. This was more stable in my VM because Spark and Cassandra were running on the same machine.

I also disabled Spark event logging because the sandbox may produce unnecessary HDFS event log messages. This setting does not change the analytical results. It only reduces environment-related logging problems.

In [ ]:
#8.start spark
spark = SparkSession.builder \
    .appName("a2_ml_spark_cassandra") \
    .master("local[*]") \
    .config("spark.cassandra.connection.host", "127.0.0.1") \
    .config("spark.cassandra.connection.port", "9042") \
    .config("spark.eventLog.enabled", "false") \
    .config("spark.ui.showConsoleProgress", "false") \
    .getOrCreate()

sc = spark.sparkContext
sc.setLogLevel("ERROR")

print("\nstart assignment 2 pipeline")
print("spark version:", spark.version)
print("hdfs path:", hdfs_path)
print("cassandra keyspace:", ks)

## 6. Read Raw Files from HDFS and Create RDDs

This step directly satisfies the RDD requirement in the assignment.

Instead of loading the files directly as DataFrames, I first use:

```python
sc.textFile()
```

This is the same direction as the classroom RDD examples.

After reading the raw files, I parse each line using the functions defined above.

In [ ]:
#9.read raw files from hdfs
print("\nread raw files from hdfs")

rate_raw = sc.textFile(rate_path)
mv_raw = sc.textFile(mv_path)
usr_raw = sc.textFile(usr_path)

print("u.data rows:", rate_raw.count())
print("u.item rows:", mv_raw.count())
print("u.user rows:", usr_raw.count())


#10.parse rdd
print("\nparse rdd")

rate_rdd = rate_raw.map(get_rate).filter(lambda x: x is not None)
mv_rdd = mv_raw.map(get_mv).filter(lambda x: x is not None)
usr_rdd = usr_raw.map(get_usr).filter(lambda x: x is not None)
mg_rdd = mv_raw.flatMap(get_mg)

## 7. Convert RDDs into Spark DataFrames

After parsing, I convert the RDDs into DataFrames.

This gives each dataset a schema and makes it easier to use Spark SQL, joins, grouping, and ordering.

This step also matches the assignment requirement to transform RDDs into Spark DataFrames.

In [ ]:
#11.make dataframe
print("\nmake dataframe")

rate_df = spark.createDataFrame(rate_rdd)
mv_df = spark.createDataFrame(mv_rdd)
usr_df = spark.createDataFrame(usr_rdd)
mg_df = spark.createDataFrame(mg_rdd)

print("\nrate_df schema")
rate_df.printSchema()

print("\nmv_df schema")
mv_df.printSchema()

print("\nusr_df schema")
usr_df.printSchema()

print("\nmg_df schema")
mg_df.printSchema()

## 8. Data Cleaning and Preprocessing

MovieLens 100k is already fairly clean, but I still add basic checks before analysis.
This is to avoid invalid or unmatched records affecting the final results.

- Drop nulls and duplicate rows.
- Filter out invalid IDs and ratings outside [1, 5].
- Inner join ratings with valid users and movies, so no orphan rating records are kept.
- Cache the cleaned DataFrames because they are reused in several tasks.

In [ ]:
#12.clean data
print("\nclean data")

rate_df = rate_df.dropna().dropDuplicates() \
    .filter((fn.col("user_id") >= 1) &
            (fn.col("movie_id") >= 1) &
            (fn.col("rating") >= 1) &
            (fn.col("rating") <= 5))

mv_df = mv_df.dropna().dropDuplicates(["movie_id"]) \
    .filter(fn.col("movie_id") >= 1)

usr_df = usr_df.dropna().dropDuplicates(["user_id"]) \
    .filter((fn.col("user_id") >= 1) & (fn.col("age") > 0))

mg_df = mg_df.dropna().dropDuplicates(["movie_id", "genre"]) \
    .filter(fn.col("movie_id") >= 1)

#keep only valid user and valid movie
rate_df = rate_df.join(usr_df.select("user_id"), "user_id", "inner") \
    .join(mv_df.select("movie_id"), "movie_id", "inner")

rate_df.cache()
mv_df.cache()
usr_df.cache()
mg_df.cache()

print("clean rating:", rate_df.count())
print("clean movie:", mv_df.count())
print("clean user:", usr_df.count())
print("movie genre rows:", mg_df.count())

print("\nsample rating")
rate_df.show(5, False)

print("\nsample movie")
mv_df.show(5, False)

print("\nsample user")
usr_df.show(5, False)

In [ ]:
#13.make temp views
rate_df.createOrReplaceTempView("rate")
mv_df.createOrReplaceTempView("mv")
usr_df.createOrReplaceTempView("usr")
mg_df.createOrReplaceTempView("mg")

## 9. Task i: Average Rating for Each Movie

Group ratings by movie, calculate the average, and join with movie titles.

I also keep `rating_count` here. Average rating alone is misleading without knowing how many users rated the movie.

In [ ]:
#14.task 1: average rating for each movie
print("\ntask 1: average rating for each movie")

avg_df = rate_df.groupBy("movie_id") \
    .agg(
        fn.round(fn.avg("rating"), 4).alias("avg_rating"),
        fn.count("rating").cast("int").alias("rating_count")
    ) \
    .join(mv_df, "movie_id", "left") \
    .select("movie_id", "title", "avg_rating", "rating_count") \
    .orderBy("movie_id")

avg_df.show(10, False)

Each row is one movie, with its average rating and rating count.

A movie with average 5.0 but only 1 rating is not as reliable as a movie with a slightly lower average but hundreds of ratings. So I keep both columns for interpretation.

## 10. Task ii: Top 10 Movies by Average Rating

Rank movies by `avg_rating` descending and show only the top 10.

I also use `rating_count` and `title` as tiebreakers, so the Spark ranking output stays consistent when average ratings are equal.

In [ ]:
#15.task 2: top 10 movies
print("\ntask 2: top 10 movies by average rating")

top_w = Window.orderBy(
    fn.desc("avg_rating"),
    fn.desc("rating_count"),
    fn.asc("title")
)

top10_df = avg_df.withColumn("rank", fn.row_number().over(top_w)) \
    .filter(fn.col("rank") <= 10) \
    .select("rank", "movie_id", "title", "avg_rating", "rating_count") \
    .orderBy("rank")

top10_df.show(10, False)

Some top-ranked movies have very few ratings but a perfect 5.0 average.

The ranking is technically correct, but `rating_count` shows how much to trust each result.

Evidence screenshot:

```text
screenshots/06_task2_top10_movies.png
```

![Task 2 top ten movies](screenshots/06_task2_top10_movies.png)

## 11. Task iii: Favourite Genre for Active Users (≥ 50 ratings)

The idea is to find users who rated at least 50 movies, then identify which genre they rated most often.

Steps:

- count total ratings per user;
- keep users with at least 50 ratings;
- join ratings with movie genres;
- count genre frequency per user;
- pick the top genre for each user.

One movie can have multiple genres, so one rating can count toward multiple genres. This follows the structure of the original `u.item` file.

In [ ]:
#16.task 3: favourite genre
print("\ntask 3: favourite genre for users with at least 50 ratings")

act_df = rate_df.groupBy("user_id") \
    .agg(fn.count("*").cast("int").alias("total_ratings")) \
    .filter(fn.col("total_ratings") >= 50)

rm_df = rate_df.join(
    mg_df.select("movie_id", "genre"),
    "movie_id",
    "inner"
)

ug_df = rm_df.groupBy("user_id", "genre") \
    .agg(fn.count("*").cast("int").alias("genre_count"))

g_w = Window.partitionBy("user_id") \
    .orderBy(fn.desc("genre_count"), fn.asc("genre"))

fav_df = ug_df.withColumn("grank", fn.row_number().over(g_w)) \
    .filter(fn.col("grank") == 1) \
    .drop("grank") \
    .join(act_df, "user_id", "inner") \
    .join(usr_df, "user_id", "left") \
    .select(
        "user_id",
        "age",
        "gender",
        "occupation",
        "zip",
        "total_ratings",
        fn.col("genre").alias("favourite_genre"),
        "genre_count"
    ) \
    .orderBy("user_id")

fav_df.show(20, False)

Most active users have `Drama` as their top genre, while some users show `Comedy`, `Action`, or `Thriller`.

Since one movie can belong to multiple genres, one rating may contribute to more than one genre count. This is expected because the dataset stores genres as multiple binary flags.

Evidence screenshot:

```text
screenshots/07_task3_fav_genre.png
```

![Task 3 favourite genre](screenshots/07_task3_fav_genre.png)

## 12. Task iv: Users Under 20

This is a simple filter on the user table by age.

I use Spark SQL here because the condition is direct and easy to read.

In [ ]:
#17.task 4: users below 20
print("\ntask 4: users less than 20 years old")

u20_df = spark.sql("""
    SELECT user_id, age, gender, occupation, zip
    FROM usr
    WHERE age < 20
    ORDER BY user_id
""")

print("number of users under 20:", u20_df.count())
u20_df.show(20, False)

This query returns users whose age is less than 20.

In my run, there were 77 users under 20. Many users in this group have occupation listed as `student`.

Evidence screenshot:

```text
screenshots/08a_task4_users_under_20.png
```

## 13. Task v: Scientists Aged 30–40

This task filters by occupation and age range together.

I use `lower(occupation)` to avoid possible casing issues in the occupation field.

In [ ]:
#18.task 5: scientist age 30 to 40
print("\ntask 5: scientist users aged 30 to 40")

sci_df = spark.sql("""
    SELECT user_id, age, gender, occupation, zip
    FROM usr
    WHERE lower(occupation) = 'scientist'
      AND age >= 30
      AND age <= 40
    ORDER BY user_id
""")

print("number of scientist age 30-40:", sci_df.count())
sci_df.show(50, False)

This query returns users whose occupation is `scientist` and whose age is from 30 to 40 inclusive.

In my run, 16 users matched both conditions.

Evidence screenshot:

```text
screenshots/08b_task5_scientists_30_40.png
```

## 14. Write Results to Cassandra

Write all five result DataFrames into Cassandra.

Need to cast some columns first. Spark may infer counts and IDs as `long`, but the Cassandra tables use `int`.

In [ ]:
#19.write result to cassandra
print("\nwrite result to cassandra")

#make cassandra type match
avg_cas = avg_df.select(
    fn.col("movie_id").cast("int").alias("movie_id"),
    "title",
    fn.col("avg_rating").cast("double").alias("avg_rating"),
    fn.col("rating_count").cast("int").alias("rating_count")
)

top10_cas = top10_df.select(
    fn.col("rank").cast("int").alias("rank"),
    fn.col("movie_id").cast("int").alias("movie_id"),
    "title",
    fn.col("avg_rating").cast("double").alias("avg_rating"),
    fn.col("rating_count").cast("int").alias("rating_count")
)

fav_cas = fav_df.select(
    fn.col("user_id").cast("int").alias("user_id"),
    fn.col("age").cast("int").alias("age"),
    "gender",
    "occupation",
    "zip",
    fn.col("total_ratings").cast("int").alias("total_ratings"),
    "favourite_genre",
    fn.col("genre_count").cast("int").alias("genre_count")
)

u20_cas = u20_df.select(
    fn.col("user_id").cast("int").alias("user_id"),
    fn.col("age").cast("int").alias("age"),
    "gender",
    "occupation",
    "zip"
)

sci_cas = sci_df.select(
    fn.col("user_id").cast("int").alias("user_id"),
    fn.col("age").cast("int").alias("age"),
    "gender",
    "occupation",
    "zip"
)

def to_cassandra(df, table):
    df.write.format("org.apache.spark.sql.cassandra") \
        .mode("append") \
        .options(table=table, keyspace=ks) \
        .save()

to_cassandra(avg_cas, "average_movie_ratings")
to_cassandra(top10_cas, "top10_movies")
to_cassandra(fav_cas, "user_favourite_genres")
to_cassandra(u20_cas, "users_under_20")
to_cassandra(sci_cas, "scientists_30_40")

print("write done")

## 15. Read Back from Cassandra

Read all five tables back into Spark to verify the data was actually stored.

In [ ]:
#20.read back from cassandra
print("\nread back from cassandra for checking")

def from_cassandra(table):
    return spark.read.format("org.apache.spark.sql.cassandra") \
        .options(table=table, keyspace=ks) \
        .load()

chk_avg = from_cassandra("average_movie_ratings").orderBy("movie_id")
chk_top10 = from_cassandra("top10_movies").orderBy("rank")
chk_fav = from_cassandra("user_favourite_genres").orderBy("user_id")
chk_u20 = from_cassandra("users_under_20").orderBy("user_id")
chk_sci = from_cassandra("scientists_30_40").orderBy("user_id")

print("\naverage_movie_ratings:")
chk_avg.show(10, False)

print("\ntop10_movies:")
chk_top10.show(10, False)

print("\nuser_favourite_genres:")
chk_fav.show(20, False)

print("\nusers_under_20:")
chk_u20.show(20, False)

print("\nscientists_30_40:")
chk_sci.show(50, False)

## 16. CQL Validation

I also checked the tables directly in `cqlsh` to confirm the data is there.

```sql
USE movielens;

SELECT rank, movie_id, avg_rating, rating_count
FROM top10_movies
LIMIT 10;

SELECT user_id, total_ratings, favourite_genre, genre_count
FROM user_favourite_genres
LIMIT 10;
```

Note: CQL does not return rows in the same order as Spark ranking output. Cassandra has no guaranteed row order without a clustering key.

So Spark output is used to show the ranked results. CQL is used to confirm the data is stored.

Evidence screenshots:

```text
screenshots/09a_cassandra_top10_validation.png
screenshots/09b_cassandra_fav_genre_validation.png
```

## 17. HBase Extension (Optional)

I also stored selected results in HBase as the optional extension.

Cassandra is still the main required database in this assignment. HBase is only used as an additional NoSQL storage layer for selected outputs.

I picked Task ii and Task iii because they are the more interesting outputs:

- Task ii is a ranking result.
- Task iii is a user preference aggregation result.

Tables created:

```text
ml_top10_movies
ml_fav_genres
```

Full commands are saved in:

```text
hbase/hbase_commands.txt
```

## 18. HBase Commands

Example commands:

```ruby
create 'ml_top10_movies', 'info'
create 'ml_fav_genres', 'info'

put 'ml_top10_movies', 'rank_1', 'info:movie_id', '1189'
put 'ml_top10_movies', 'rank_1', 'info:title', 'Prefontaine (1997)'
put 'ml_top10_movies', 'rank_1', 'info:avg_rating', '5.0'
put 'ml_top10_movies', 'rank_1', 'info:rating_count', '3'

put 'ml_fav_genres', 'user_1', 'info:age', '24'
put 'ml_fav_genres', 'user_1', 'info:occupation', 'technician'
put 'ml_fav_genres', 'user_1', 'info:total_ratings', '272'
put 'ml_fav_genres', 'user_1', 'info:favourite_genre', 'Drama'
put 'ml_fav_genres', 'user_1', 'info:genre_count', '107'

scan 'ml_top10_movies', {LIMIT => 3}
scan 'ml_fav_genres', {LIMIT => 3}
```

One issue I ran into: HBase shell could not list tables at first because RegionServer was not live.

I fixed it by starting HBase properly in Ambari and waiting for both HMaster and HRegionServer to be up.

Evidence screenshots:

```text
screenshots/11a_hbase_top10_movies.png
screenshots/11b_hbase_fav_genres.png
```

![HBase top10 movies](screenshots/11a_hbase_top10_movies.png)

![HBase favourite genres](screenshots/11b_hbase_fav_genres.png)

## 19. Screenshots

Evidence for each step is saved in `screenshots/`.

```text
01_hdfs_files.png                         HDFS file check
02_versions.png                           environment versions
03_cassandra_connection.png               Cassandra connection
04_cassandra_tables.png                   keyspace and tables
05_spark_cassandra_test.png               Spark-Cassandra test
06_task2_top10_movies.png                 Task ii output
07_task3_fav_genre.png                    Task iii output
08a_task4_users_under_20.png              Task iv output
08b_task5_scientists_30_40.png            Task v output
09a_cassandra_top10_validation.png        CQL check for top10
09b_cassandra_fav_genre_validation.png    CQL check for favourite genre
11a_hbase_top10_movies.png                HBase Task ii
11b_hbase_fav_genres.png                  HBase Task iii
```

Terminal run output:

```text
logs/assignment2_run_output.txt
```

## 20. Reproducibility Notes

Things to check before running:

1. Cassandra must be running. Test with `cqlsh 127.0.0.1 9042`.
2. Run `cql/schema.cql` before Spark writes to Cassandra.
3. I used `local[*]` because Spark and Cassandra are on the same VM.
4. `spark.eventLog.enabled=false` avoids HDFS event log issues in the sandbox.
5. For HBase extension, check that both HMaster and HRegionServer are live in Ambari.
6. Spark output shows ordered results. CQL output just confirms data is stored.
7. Main executable is `assignment2_movielens.py`. This notebook is the documented version.

## 21. Summary

This pipeline covers the full workflow from raw HDFS files to Cassandra storage and read-back.

- Task i and ii: rating aggregation and ranking. I kept `rating_count` alongside average because average alone does not tell the full story.
- Task iii: genre preference for active users. More steps are needed because one movie can have multiple genres.
- Task iv and v: straightforward user profile filters.
- Cassandra stores all five results and read-back confirms the data was written.
- HBase extension stores selected Task ii and Task iii results as an additional NoSQL storage demo.

In [ ]:
#stop spark
spark.stop()
print("done")